## Feature Engineering

This Notebook creates customer-level features for cohort-analysis, clustering, product recommendations, and lead generation

Setup

In [32]:
from pathlib import Path 

import numpy as np 
import pandas as pd 

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

INPUT_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/feature_engineering")
OUTPUT_DIR.mkdir(parents=True, exist_ok = True)

print("Input directory:", INPUT_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Input directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\processed
Output directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\feature_engineering


Load cleanded data

In [33]:
df_sales_full = pd.read_csv(INPUT_DIR / "df_sales_full.csv")
fact_sf = pd.read_csv(INPUT_DIR / "fact_sf_clean.csv")
dim_product = pd.read_csv(INPUT_DIR / "dim_product_clean.csv")

df_sales_full["Date"] = pd.to_datetime(df_sales_full["Date"], errors="coerce")
fact_sf["Date"] = pd.to_datetime(fact_sf["Date"], errors="coerce")
df_sales_full["YearMonth"] = df_sales_full["Date"].dt.to_period("M")
fact_sf["YearMonth"] = fact_sf["Date"].dt.to_period("M")

display(pd.DataFrame({
    "table": ["df_sales_full", "fact_sf", "dim_product"],
    "rows": [len(df_sales_full), len(fact_sf), len(dim_product)],
    "columns": [df_sales_full.shape[1], fact_sf.shape[1], dim_product.shape[1]]
}))

,table,rows,columns
0,df_sales_full,68723,24
1,fact_sf,39508,11
2,dim_product,180,8


In [8]:
display(df_sales_full.head())
display(fact_sf.head())
display(dim_product.head())

,Date,ID_Customer,ID_Product,Sales,Units,YearMonth,Year,Month,Customer_Name,Customer_Segment,Customer_Size,Industry,Region_CC,Region_OC,Country,Acquisition_Channel,Customer_Start_Date,Product_Name,Prod_Line,Prd_Grp_1,Prd_Grp_2,Prd_Grp_3,Unit_Price,Margin_Category
0,2023-02-01,C00001,P0116,"2,301.44",13,2023-02,2023,2,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Premium Standard Type 2 0116,Consumables,Premium Products,Premium Standard,Premium Standard Type 2,194.35,Low
1,2023-02-01,C00001,P0083,"1,545.15",6,2023-02,2023,2,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Premium Plus Type 1 0083,Maintenance,Premium Products,Premium Plus,Premium Plus Type 1,285.90,High
2,2023-04-01,C00001,P0160,763.50,15,2023-04,2023,4,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Repair Kits Type 1 0160,Industrial Equipment,Maintenance Kits,Repair Kits,Repair Kits Type 1,54.56,Low
3,2023-05-01,C00001,P0170,805.03,19,2023-05,2023,5,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Core Basic Type 2 0170,Industrial Equipment,Core Products,Core Basic,Core Basic Type 2,42.54,Medium
4,2023-05-01,C00001,P0160,155.10,3,2023-05,2023,5,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Repair Kits Type 1 0160,Industrial Equipment,Maintenance Kits,Repair Kits,Repair Kits Type 1,54.56,Low


,Date,ID_Customer,SF_Activity_Count,SF_Selling_Time,SF_Activity_Time_,Activity_Type,Sales_Rep,Opportunity_Stage,YearMonth,Year,Month
0,2023-01-01,C00001,1,0.51,0.38,Account Review,Rep_13,Prospecting,2023-01,2023,1
1,2023-02-01,C00001,1,0.74,0.36,Call,Rep_02,Proposal,2023-02,2023,2
2,2023-03-01,C00001,2,2.28,0.58,Account Review,Rep_14,Closed Lost,2023-03,2023,3
3,2023-05-01,C00001,12,8.09,7.56,Call,Rep_21,Proposal,2023-05,2023,5
4,2023-06-01,C00001,4,4.11,1.58,Demo,Rep_01,Qualification,2023-06,2023,6


,ID_Product,Product_Name,Prod_Line,Prd_Grp_1,Prd_Grp_2,Prd_Grp_3,Unit_Price,Margin_Category
0,P0001,Controllers Type 1 0001,Safety,Automation Modules,Controllers,Controllers Type 1,583.74,Low
1,P0002,Extended Support Type 2 0002,Specialty Materials,Service Contracts,Extended Support,Extended Support Type 2,581.89,Medium
2,P0003,Repair Kits Type 1 0003,Consumables,Maintenance Kits,Repair Kits,Repair Kits Type 1,122.81,High
3,P0004,Calibration Kits Type 1 0004,Consumables,Maintenance Kits,Calibration Kits,Calibration Kits Type 1,136.53,Low
4,P0005,Sensors Type 2 0005,Consumables,Automation Modules,Sensors,Sensors Type 2,330.94,High


Helper Functions

In [ ]:
def month_ordinal(period_series: pd.Series) -> pd.Series:
    return period_series.astype("period[M]").astype(int)


def streak_gap_stats(month_ordinals: np.ndarray) -> dict:
    months = np.sort(np.unique(month_ordinals))
    if len(months) == 0:
        return {
            "max_consec_active_months": 0,
            "current_consec_active_months": 0,
            "avg_inactive_gap_months": 0.0,
            "max_inactive_gap_months": 0}
    if len(months) == 1:
        return {
            "max_consec_active_months": 1,
            "current_consec_active_months": 1,
            "avg_inactive_gap_months": 0.0,
            "max_inactive_gap_months": 0}

    diffs = np.diff(months)
    gaps = np.maximum(diffs - 1, 0)

    run = 1
    max_run = 1
    for diff in diffs:
        if diff == 1:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 1

    current_run = 1
    for diff in diffs[::-1]:
        if diff == 1:
            current_run += 1
        else:
            break

    return {
        "max_consec_active_months": int(max_run),
        "current_consec_active_months": int(current_run),
        "avg_inactive_gap_months": float(np.mean(gaps)),
        "max_inactive_gap_months": int(np.max(gaps))}


def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)


Product Hierarchy Cardinality

In [ ]:
product_cardinality = pd.DataFrame({
    "level": ["Prod_Line", "Prd_Grp_1", "Prd_Grp_2", "Prd_Grp_3", "ID_Product"],
    "unique_values": [
        dim_product["Prod_Line"].nunique(dropna=True),
        dim_product["Prd_Grp_1"].nunique(dropna=True),
        dim_product["Prd_Grp_2"].nunique(dropna=True),
        dim_product["Prd_Grp_3"].nunique(dropna=True),
        dim_product["ID_Product"].nunique(dropna=True)]})

display(product_cardinality)


,level,unique_values
0,Prod_Line,7
1,Prd_Grp_1,6
2,Prd_Grp_2,18
3,Prd_Grp_3,35
4,ID_Product,180


Customer Sales Aggregations

In [ ]:
unique_prd_line = dim_product["Prod_Line"].nunique(dropna=True)
unique_prd_grp_1 = dim_product["Prd_Grp_1"].nunique(dropna=True)
unique_prd_grp_2 = dim_product["Prd_Grp_2"].nunique(dropna=True)
unique_prd_grp_3 = dim_product["Prd_Grp_3"].nunique(dropna=True)

customer_sales_agg = (
    df_sales_full
    .groupby("ID_Customer", as_index=False)
    .agg(
        Sales=("Sales", "sum"),
        Units=("Units", "sum"),
        order_lines=("ID_Product", "count"),
        active_purchase_months=("YearMonth", "nunique"),
        distinct_products=("ID_Product", "nunique"),
        prd_line_coverage=("Prod_Line", "nunique"),
        prd_grp_1_coverage=("Prd_Grp_1", "nunique"),
        prd_grp_2_coverage=("Prd_Grp_2", "nunique"),
        prd_grp_3_coverage=("Prd_Grp_3", "nunique"),
        customer_first_active_date=("Date", "min"),
        customer_last_active_date=("Date", "max")))

customer_sales_agg["prd_line_coverage_pct"] = customer_sales_agg["prd_line_coverage"] / unique_prd_line
customer_sales_agg["prd_grp_1_coverage_pct"] = customer_sales_agg["prd_grp_1_coverage"] / unique_prd_grp_1
customer_sales_agg["prd_grp_2_coverage_pct"] = customer_sales_agg["prd_grp_2_coverage"] / unique_prd_grp_2
customer_sales_agg["prd_grp_3_coverage_pct"] = customer_sales_agg["prd_grp_3_coverage"] / unique_prd_grp_3

customer_sales_agg["avg_order_line_value"] = safe_divide(customer_sales_agg["Sales"], customer_sales_agg["order_lines"]).fillna(0)
customer_sales_agg["avg_unit_price_realized"] = safe_divide(customer_sales_agg["Sales"], customer_sales_agg["Units"]).fillna(0)

display(customer_sales_agg.head())
display(customer_sales_agg.describe(include="all").T.head(20))


,ID_Customer,Sales,Units,order_lines,active_purchase_months,distinct_products,prd_line_coverage,prd_grp_1_coverage,prd_grp_2_coverage,prd_grp_3_coverage,customer_first_active_date,customer_last_active_date,prd_line_coverage_pct,prd_grp_1_coverage_pct,prd_grp_2_coverage_pct,prd_grp_3_coverage_pct,avg_order_line_value,avg_unit_price_realized
0,C00001,"144,433.27",653,47,23,5,4,4,5,5,2023-02-01,2025-11-01,0.57,0.67,0.28,0.14,"3,073.05",221.18
1,C00002,"4,738.61",21,17,8,9,7,4,6,8,2023-05-01,2025-11-01,1.00,0.67,0.33,0.23,278.74,225.65
2,C00003,"199,361.80",658,105,31,11,4,5,9,10,2023-01-01,2025-11-01,0.57,0.83,0.50,0.29,"1,898.68",302.98
3,C00004,"130,129.62",684,76,26,4,4,3,4,4,2023-01-01,2025-12-01,0.57,0.50,0.22,0.11,"1,712.23",190.25
4,C00005,"39,122.45",78,36,17,5,5,4,4,5,2023-02-01,2025-07-01,0.71,0.67,0.22,0.14,"1,086.73",501.57


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ID_Customer,1993,1993,C00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sales,"1,993.00",NaN,NaN,NaN,"66,171.92",43.51,"7,366.84","19,344.95","59,973.28","1,524,833.59","132,526.71"
Units,"1,993.00",NaN,NaN,NaN,235.07,1.00,32.00,75.00,231.00,"3,174.00",418.64
order_lines,"1,993.00",NaN,NaN,NaN,34.48,1.00,14.00,23.00,36.00,127.00,31.50
active_purchase_months,"1,993.00",NaN,NaN,NaN,12.87,1.00,6.00,10.00,16.00,36.00,8.71
distinct_products,"1,993.00",NaN,NaN,NaN,7.38,1.00,5.00,7.00,9.00,13.00,2.69
prd_line_coverage,"1,993.00",NaN,NaN,NaN,4.58,1.00,4.00,5.00,5.00,7.00,1.25
prd_grp_1_coverage,"1,993.00",NaN,NaN,NaN,3.88,1.00,3.00,4.00,5.00,6.00,1.07
prd_grp_2_coverage,"1,993.00",NaN,NaN,NaN,5.78,1.00,4.00,6.00,7.00,13.00,1.89
prd_grp_3_coverage,"1,993.00",NaN,NaN,NaN,6.52,1.00,5.00,6.00,8.00,13.00,2.22


Customer activity, Recency and Inactivity Gaps 

In [ ]:
sales_monthly = (
            df_sales_full
            .drop_duplicates(["ID_Customer", "YearMonth"])
            .sort_values(["ID_Customer", "YearMonth"])
            .assign(YM_ORD=lambda d: month_ordinal(d["YearMonth"])))

max_sales_ord = sales_monthly["YM_ORD"].max()

recent_flags = (
    sales_monthly
    .groupby("ID_Customer")
    .agg(
        first_YM=("YearMonth", "min"),
        last_YM=("YearMonth", "max"),
        first_YM_ORD=("YM_ORD", "min"),
        last_YM_ORD=("YM_ORD", "max"),
        active_months=("YearMonth", "nunique"),
        active_in_last_3m=("YM_ORD", lambda s: int((s >= max_sales_ord - 3).any())),
        active_in_last_6m=("YM_ORD", lambda s: int((s >= max_sales_ord - 6).any())),
        active_in_last_12m=("YM_ORD", lambda s: int((s >= max_sales_ord - 12).any())))
    .reset_index())

gap_features = (
    sales_monthly
    .groupby("ID_Customer")["YM_ORD"]
    .apply(lambda s: pd.Series(streak_gap_stats(s.to_numpy())))
    .unstack()
    .reset_index())

customer_activity = recent_flags.merge(gap_features, on="ID_Customer", how="left")
customer_activity["lifetime_months"] = (
customer_activity["last_YM_ORD"] - customer_activity["first_YM_ORD"] + 1).clip(lower=1)
customer_activity["months_since_last_purchase"] = (
    max_sales_ord - customer_activity["last_YM_ORD"]).clip(lower=0)
customer_activity["activity_ratio_lifetime"] = (
    customer_activity["active_months"] / customer_activity["lifetime_months"]).clip(0, 1)
customer_activity["is_inactive_6m"] = (customer_activity["months_since_last_purchase"] >= 6).astype(int)
customer_activity["is_inactive_12m"] = (customer_activity["months_since_last_purchase"] >= 12).astype(int)

display(customer_activity.head())
display(customer_activity[[
    "active_months", "lifetime_months", "months_since_last_purchase",
    "activity_ratio_lifetime", "avg_inactive_gap_months", "max_inactive_gap_months"
]].describe().T)


,ID_Customer,first_YM,last_YM,first_YM_ORD,last_YM_ORD,active_months,active_in_last_3m,active_in_last_6m,active_in_last_12m,max_consec_active_months,current_consec_active_months,avg_inactive_gap_months,max_inactive_gap_months,lifetime_months,months_since_last_purchase,activity_ratio_lifetime,is_inactive_6m,is_inactive_12m
0,C00001,2023-02,2025-11,637,670,23,1,1,1,7.00,3.00,0.50,3.00,34,1,0.68,0,0
1,C00002,2023-05,2025-11,640,670,8,1,1,1,3.00,1.00,3.29,14.00,31,1,0.26,0,0
2,C00003,2023-01,2025-11,636,670,31,1,1,1,15.00,4.00,0.13,1.00,35,1,0.89,0,0
3,C00004,2023-01,2025-12,636,671,26,1,1,1,10.00,10.00,0.40,3.00,36,0,0.72,0,0
4,C00005,2023-02,2025-07,637,666,17,0,1,1,5.00,1.00,0.81,5.00,30,5,0.57,0,0


,count,mean,std,min,25%,50%,75%,max
active_months,"1,993.00",12.87,8.71,1.00,6.00,10.00,16.00,36.00
lifetime_months,"1,993.00",26.57,9.06,1.00,22.00,29.00,34.00,36.00
months_since_last_purchase,"1,993.00",5.25,7.97,0.00,0.00,1.00,6.00,35.00
activity_ratio_lifetime,"1,993.00",0.48,0.23,0.07,0.30,0.43,0.67,1.00
avg_inactive_gap_months,"1,993.00",2.13,2.41,0.00,0.53,1.50,2.75,28.00
max_inactive_gap_months,"1,993.00",6.21,4.86,0.00,3.00,5.00,8.00,30.00


RFM and Value Features 

In [18]:
customer_features = customer_sales_agg.merge(customer_activity, on="ID_Customer", how="left")

customer_features["avg_monthly_revenue"] = safe_divide(
    customer_features["Sales"], customer_features["lifetime_months"]).fillna(0)

# Historical LTV proxy: in this synthetic project this is historical value, not predictive CLV.
customer_features["historical_ltv"] = customer_features["Sales"]

customer_features["purchase_freq_per_year"] = (
    customer_features["active_months"] / (customer_features["lifetime_months"] / 12).replace(0, np.nan)
).fillna(0)

customer_features["R_recency_m"] = customer_features["months_since_last_purchase"]
customer_features["F_frequency_m"] = customer_features["active_months"]
customer_features["M_sales"] = customer_features["Sales"]

customer_features["sales_value_segment"] = pd.qcut(
    customer_features["Sales"],
    q=4,
    labels=["Low Value", "Mid Value", "High Value", "Top Value"],
    duplicates="drop")

customer_features["recency_segment"] = pd.cut(
    customer_features["months_since_last_purchase"],
    bins=[-1, 1, 3, 6, 99],
    labels=["Very Recent", "Recent", "Cooling", "Inactive"])

display(customer_features.head())
display(customer_features[["sales_value_segment", "recency_segment"]].value_counts().reset_index(name="customers"))


,ID_Customer,Sales,Units,order_lines,active_purchase_months,distinct_products,prd_line_coverage,prd_grp_1_coverage,prd_grp_2_coverage,prd_grp_3_coverage,customer_first_active_date,customer_last_active_date,prd_line_coverage_pct,prd_grp_1_coverage_pct,prd_grp_2_coverage_pct,prd_grp_3_coverage_pct,avg_order_line_value,avg_unit_price_realized,first_YM,last_YM,first_YM_ORD,last_YM_ORD,active_months,active_in_last_3m,active_in_last_6m,active_in_last_12m,max_consec_active_months,current_consec_active_months,avg_inactive_gap_months,max_inactive_gap_months,lifetime_months,months_since_last_purchase,activity_ratio_lifetime,is_inactive_6m,is_inactive_12m,avg_monthly_revenue,historical_ltv,purchase_freq_per_year,R_recency_m,F_frequency_m,M_sales,sales_value_segment,recency_segment
0,C00001,"144,433.27",653,47,23,5,4,4,5,5,2023-02-01,2025-11-01,0.57,0.67,0.28,0.14,"3,073.05",221.18,2023-02,2025-11,637,670,23,1,1,1,7.00,3.00,0.50,3.00,34,1,0.68,0,0,"4,248.04","144,433.27",8.12,1,23,"144,433.27",Top Value,Very Recent
1,C00002,"4,738.61",21,17,8,9,7,4,6,8,2023-05-01,2025-11-01,1.00,0.67,0.33,0.23,278.74,225.65,2023-05,2025-11,640,670,8,1,1,1,3.00,1.00,3.29,14.00,31,1,0.26,0,0,152.86,"4,738.61",3.10,1,8,"4,738.61",Low Value,Very Recent
2,C00003,"199,361.80",658,105,31,11,4,5,9,10,2023-01-01,2025-11-01,0.57,0.83,0.50,0.29,"1,898.68",302.98,2023-01,2025-11,636,670,31,1,1,1,15.00,4.00,0.13,1.00,35,1,0.89,0,0,"5,696.05","199,361.80",10.63,1,31,"199,361.80",Top Value,Very Recent
3,C00004,"130,129.62",684,76,26,4,4,3,4,4,2023-01-01,2025-12-01,0.57,0.50,0.22,0.11,"1,712.23",190.25,2023-01,2025-12,636,671,26,1,1,1,10.00,10.00,0.40,3.00,36,0,0.72,0,0,"3,614.71","130,129.62",8.67,0,26,"130,129.62",Top Value,Very Recent
4,C00005,"39,122.45",78,36,17,5,5,4,4,5,2023-02-01,2025-07-01,0.71,0.67,0.22,0.14,"1,086.73",501.57,2023-02,2025-07,637,666,17,0,1,1,5.00,1.00,0.81,5.00,30,5,0.57,0,0,"1,304.08","39,122.45",6.80,5,17,"39,122.45",High Value,Cooling


,sales_value_segment,recency_segment,customers
0,Top Value,Very Recent,420
1,High Value,Very Recent,261
2,Low Value,Inactive,252
3,Mid Value,Very Recent,201
4,Mid Value,Inactive,131
5,Low Value,Very Recent,122
6,Mid Value,Recent,98
7,High Value,Inactive,96
8,High Value,Recent,80
9,Low Value,Recent,71


Salesforce Activity Features

In [20]:
sf_monthly = (
    fact_sf
    .drop_duplicates(["ID_Customer", "YearMonth"])
    .sort_values(["ID_Customer", "YearMonth"])
    .assign(SF_YM_ORD=lambda d: month_ordinal(d["YearMonth"])))

max_sf_ord = sf_monthly["SF_YM_ORD"].max()

sf_recent_flags = (
    sf_monthly
    .groupby("ID_Customer")
    .agg(
        sf_first_YM=("YearMonth", "min"),
        sf_last_YM=("YearMonth", "max"),
        sf_first_YM_ORD=("SF_YM_ORD", "min"),
        sf_last_YM_ORD=("SF_YM_ORD", "max"),
        sf_active_months=("YearMonth", "nunique"),
        sf_active_in_last_3m=("SF_YM_ORD", lambda s: int((s >= max_sf_ord - 3).any())),
        sf_active_in_last_6m=("SF_YM_ORD", lambda s: int((s >= max_sf_ord - 6).any())),
        sf_active_in_last_12m=("SF_YM_ORD", lambda s: int((s >= max_sf_ord - 12).any())))
    .reset_index())

sf_gap_features = (
    sf_monthly
    .groupby("ID_Customer")["SF_YM_ORD"]
    .apply(lambda s: pd.Series(streak_gap_stats(s.to_numpy())))
    .unstack()
    .reset_index()
    .rename(columns={
        "max_consec_active_months": "sf_max_consec_active_months",
        "current_consec_active_months": "sf_current_consec_active_months",
        "avg_inactive_gap_months": "sf_avg_inactive_gap_months",
        "max_inactive_gap_months": "sf_max_inactive_gap_months"}))

sf_summary = (
    fact_sf
    .groupby("ID_Customer", as_index=False)
    .agg(
        sf_total_events=("SF_Activity_Count", "sum"),
        sf_total_selling_time=("SF_Selling_Time", "sum"),
        sf_total_activity_time=("SF_Activity_Time_", "sum"),
        sf_distinct_activity_types=("Activity_Type", "nunique"),
        sf_distinct_reps=("Sales_Rep", "nunique")))

sf_features = (
    sf_recent_flags
    .merge(sf_gap_features, on="ID_Customer", how="left")
    .merge(sf_summary, on="ID_Customer", how="left"))

sf_features["sf_lifetime_months"] = (
    sf_features["sf_last_YM_ORD"] - sf_features["sf_first_YM_ORD"] + 1).clip(lower=1)
sf_features["sf_months_since_last_event"] = (
    max_sf_ord - sf_features["sf_last_YM_ORD"]).clip(lower=0)
sf_features["sf_activity_ratio_lifetime"] = (
    sf_features["sf_active_months"] / sf_features["sf_lifetime_months"]).clip(0, 1)
sf_features["sf_events_per_active_month"] = safe_divide(
    sf_features["sf_total_events"], sf_features["sf_active_months"]).fillna(0)

display(sf_features.head())
display(sf_features.describe(include="all").T.head(25))


,ID_Customer,sf_first_YM,sf_last_YM,sf_first_YM_ORD,sf_last_YM_ORD,sf_active_months,sf_active_in_last_3m,sf_active_in_last_6m,sf_active_in_last_12m,sf_max_consec_active_months,sf_current_consec_active_months,sf_avg_inactive_gap_months,sf_max_inactive_gap_months,sf_total_events,sf_total_selling_time,sf_total_activity_time,sf_distinct_activity_types,sf_distinct_reps,sf_lifetime_months,sf_months_since_last_event,sf_activity_ratio_lifetime,sf_events_per_active_month
0,C00001,2023-01,2025-12,636,671,29,1,1,1,10.00,1.00,0.25,1.00,117,81.92,67.59,5,15,36,0,0.81,4.03
1,C00002,2023-06,2025-11,641,670,13,1,1,1,4.00,2.00,1.42,9.00,45,28.87,18.84,3,10,30,1,0.43,3.46
2,C00003,2023-01,2025-09,636,668,24,1,1,1,8.00,1.00,0.39,3.00,124,76.30,70.78,6,15,33,3,0.73,5.17
3,C00004,2023-01,2025-11,636,670,23,1,1,1,7.00,1.00,0.55,5.00,100,59.66,48.33,6,16,35,1,0.66,4.35
4,C00005,2023-01,2025-12,636,671,23,1,1,1,5.00,1.00,0.59,4.00,82,47.96,43.34,5,15,36,0,0.64,3.57


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID_Customer,2000,2000,C00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sf_first_YM,2000,14,2023-01,1118,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sf_last_YM,2000,18,2025-12,1128,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sf_first_YM_ORD,"2,000.00",NaN,NaN,NaN,637.04,1.76,636.00,636.00,636.00,637.00,652.00
sf_last_YM_ORD,"2,000.00",NaN,NaN,NaN,669.94,1.99,652.00,670.00,671.00,671.00,671.00
sf_active_months,"2,000.00",NaN,NaN,NaN,19.75,6.96,4.00,14.00,21.00,25.00,35.00
sf_active_in_last_3m,"2,000.00",NaN,NaN,NaN,0.91,0.28,0.00,1.00,1.00,1.00,1.00
sf_active_in_last_6m,"2,000.00",NaN,NaN,NaN,0.97,0.17,0.00,1.00,1.00,1.00,1.00
sf_active_in_last_12m,"2,000.00",NaN,NaN,NaN,1.00,0.07,0.00,1.00,1.00,1.00,1.00
sf_max_consec_active_months,"2,000.00",NaN,NaN,NaN,5.88,3.66,1.00,3.00,5.00,8.00,35.00


Combine Customer and Salesforce Features 

In [22]:
dim_customer_features = customer_features.merge(sf_features, on="ID_Customer", how="left")

sf_fill_zero_cols = [
    c for c in dim_customer_features.columns
    if c.startswith("sf_") and pd.api.types.is_numeric_dtype(dim_customer_features[c])]
dim_customer_features[sf_fill_zero_cols] = dim_customer_features[sf_fill_zero_cols].fillna(0)

dim_customer_features["has_salesforce_activity"] = (dim_customer_features["sf_total_events"] > 0).astype(int)
dim_customer_features["sales_without_recent_sf_activity"] = (
    (dim_customer_features["active_in_last_6m"] == 1)
    & (dim_customer_features["sf_active_in_last_6m"].fillna(0) == 0)
).astype(int)
dim_customer_features["sf_activity_without_recent_sales"] = (
    (dim_customer_features["sf_active_in_last_6m"].fillna(0) == 1)
    & (dim_customer_features["active_in_last_6m"] == 0)
).astype(int)

display(dim_customer_features.head())
display(dim_customer_features[[
    "has_salesforce_activity",
    "sales_without_recent_sf_activity",
    "sf_activity_without_recent_sales"
]].sum().reset_index(name="customers"))


,ID_Customer,Sales,Units,order_lines,active_purchase_months,distinct_products,prd_line_coverage,prd_grp_1_coverage,prd_grp_2_coverage,prd_grp_3_coverage,customer_first_active_date,customer_last_active_date,prd_line_coverage_pct,prd_grp_1_coverage_pct,prd_grp_2_coverage_pct,prd_grp_3_coverage_pct,avg_order_line_value,avg_unit_price_realized,first_YM,last_YM,first_YM_ORD,last_YM_ORD,active_months,active_in_last_3m,active_in_last_6m,active_in_last_12m,max_consec_active_months,current_consec_active_months,avg_inactive_gap_months,max_inactive_gap_months,lifetime_months,months_since_last_purchase,activity_ratio_lifetime,is_inactive_6m,is_inactive_12m,avg_monthly_revenue,historical_ltv,purchase_freq_per_year,R_recency_m,F_frequency_m,M_sales,sales_value_segment,recency_segment,sf_first_YM,sf_last_YM,sf_first_YM_ORD,sf_last_YM_ORD,sf_active_months,sf_active_in_last_3m,sf_active_in_last_6m,sf_active_in_last_12m,sf_max_consec_active_months,sf_current_consec_active_months,sf_avg_inactive_gap_months,sf_max_inactive_gap_months,sf_total_events,sf_total_selling_time,sf_total_activity_time,sf_distinct_activity_types,sf_distinct_reps,sf_lifetime_months,sf_months_since_last_event,sf_activity_ratio_lifetime,sf_events_per_active_month,has_salesforce_activity,sales_without_recent_sf_activity,sf_activity_without_recent_sales
0,C00001,"144,433.27",653,47,23,5,4,4,5,5,2023-02-01,2025-11-01,0.57,0.67,0.28,0.14,"3,073.05",221.18,2023-02,2025-11,637,670,23,1,1,1,7.00,3.00,0.50,3.00,34,1,0.68,0,0,"4,248.04","144,433.27",8.12,1,23,"144,433.27",Top Value,Very Recent,2023-01,2025-12,636,671,29,1,1,1,10.00,1.00,0.25,1.00,117,81.92,67.59,5,15,36,0,0.81,4.03,1,0,0
1,C00002,"4,738.61",21,17,8,9,7,4,6,8,2023-05-01,2025-11-01,1.00,0.67,0.33,0.23,278.74,225.65,2023-05,2025-11,640,670,8,1,1,1,3.00,1.00,3.29,14.00,31,1,0.26,0,0,152.86,"4,738.61",3.10,1,8,"4,738.61",Low Value,Very Recent,2023-06,2025-11,641,670,13,1,1,1,4.00,2.00,1.42,9.00,45,28.87,18.84,3,10,30,1,0.43,3.46,1,0,0
2,C00003,"199,361.80",658,105,31,11,4,5,9,10,2023-01-01,2025-11-01,0.57,0.83,0.50,0.29,"1,898.68",302.98,2023-01,2025-11,636,670,31,1,1,1,15.00,4.00,0.13,1.00,35,1,0.89,0,0,"5,696.05","199,361.80",10.63,1,31,"199,361.80",Top Value,Very Recent,2023-01,2025-09,636,668,24,1,1,1,8.00,1.00,0.39,3.00,124,76.30,70.78,6,15,33,3,0.73,5.17,1,0,0
3,C00004,"130,129.62",684,76,26,4,4,3,4,4,2023-01-01,2025-12-01,0.57,0.50,0.22,0.11,"1,712.23",190.25,2023-01,2025-12,636,671,26,1,1,1,10.00,10.00,0.40,3.00,36,0,0.72,0,0,"3,614.71","130,129.62",8.67,0,26,"130,129.62",Top Value,Very Recent,2023-01,2025-11,636,670,23,1,1,1,7.00,1.00,0.55,5.00,100,59.66,48.33,6,16,35,1,0.66,4.35,1,0,0
4,C00005,"39,122.45",78,36,17,5,5,4,4,5,2023-02-01,2025-07-01,0.71,0.67,0.22,0.14,"1,086.73",501.57,2023-02,2025-07,637,666,17,0,1,1,5.00,1.00,0.81,5.00,30,5,0.57,0,0,"1,304.08","39,122.45",6.80,5,17,"39,122.45",High Value,Cooling,2023-01,2025-12,636,671,23,1,1,1,5.00,1.00,0.59,4.00,82,47.96,43.34,5,15,36,0,0.64,3.57,1,0,0


,index,customers
0,has_salesforce_activity,1993
1,sales_without_recent_sf_activity,40
2,sf_activity_without_recent_sales,481


Product Purchase Matrices 

In [24]:
df_sales_full["purchased_flag"] = (df_sales_full["Units"] > 0).astype(int)

def create_product_matrices(df: pd.DataFrame, product_group_col: str, prefix: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    flag_matrix = (
        df.pivot_table(
            index="ID_Customer",
            columns=product_group_col,
            values="purchased_flag",
            aggfunc="max",
            fill_value=0)
        .rename_axis(None, axis=1)
        .add_prefix(f"purchase_flag_{prefix}_")
        .reset_index())

    share_matrix = (
        df.pivot_table(
            index="ID_Customer",
            columns=product_group_col,
            values="Units",
            aggfunc="sum",
            fill_value=0)
        .pipe(lambda d: d.div(d.sum(axis=1).replace(0, np.nan), axis=0))
        .fillna(0)
        .rename_axis(None, axis=1)
        .add_prefix(f"purchase_share_{prefix}_")
        .reset_index())

    return flag_matrix, share_matrix


cust_prod_matrix_grp_1, cust_prod_matrix_grp_1_share = create_product_matrices(
    df_sales_full, "Prd_Grp_1", "grp1")
cust_prod_matrix_grp_3, cust_prod_matrix_grp_3_share = create_product_matrices(
    df_sales_full, "Prd_Grp_3", "grp3")

cust_prod_matrix_share_combined_grp_1_grp_3 = cust_prod_matrix_grp_1_share.merge(
    cust_prod_matrix_grp_3_share, on="ID_Customer", how="left")

display(cust_prod_matrix_grp_1.head())
display(cust_prod_matrix_grp_3.head())
display(cust_prod_matrix_share_combined_grp_1_grp_3.head())


,ID_Customer,purchase_flag_grp1_Automation Modules,purchase_flag_grp1_Core Products,purchase_flag_grp1_Maintenance Kits,purchase_flag_grp1_Premium Products,purchase_flag_grp1_Safety Solutions,purchase_flag_grp1_Service Contracts
0,C00001,1,1,1,1,0,0
1,C00002,1,1,0,1,1,0
2,C00003,0,1,1,1,1,1
3,C00004,1,1,1,0,0,0
4,C00005,1,1,0,1,0,1


,ID_Customer,purchase_flag_grp3_Calibration Kits Type 1,purchase_flag_grp3_Calibration Kits Type 2,purchase_flag_grp3_Compliance Safety Type 1,purchase_flag_grp3_Compliance Safety Type 2,purchase_flag_grp3_Controllers Type 1,purchase_flag_grp3_Controllers Type 2,purchase_flag_grp3_Core Advanced Type 1,purchase_flag_grp3_Core Advanced Type 2,purchase_flag_grp3_Core Basic Type 1,purchase_flag_grp3_Core Basic Type 2,purchase_flag_grp3_Core Replacement Type 1,purchase_flag_grp3_Core Replacement Type 2,purchase_flag_grp3_Extended Support Type 1,purchase_flag_grp3_Extended Support Type 2,purchase_flag_grp3_Facility Safety Type 2,purchase_flag_grp3_Installation Type 1,purchase_flag_grp3_Installation Type 2,purchase_flag_grp3_Monitoring Type 1,purchase_flag_grp3_Monitoring Type 2,purchase_flag_grp3_Personal Safety Type 1,purchase_flag_grp3_Personal Safety Type 2,purchase_flag_grp3_Premium Custom Type 1,purchase_flag_grp3_Premium Custom Type 2,purchase_flag_grp3_Premium Plus Type 1,purchase_flag_grp3_Premium Plus Type 2,purchase_flag_grp3_Premium Standard Type 1,purchase_flag_grp3_Premium Standard Type 2,purchase_flag_grp3_Preventive Kits Type 1,purchase_flag_grp3_Preventive Kits Type 2,purchase_flag_grp3_Repair Kits Type 1,purchase_flag_grp3_Repair Kits Type 2,purchase_flag_grp3_Sensors Type 1,purchase_flag_grp3_Sensors Type 2,purchase_flag_grp3_Training Type 1,purchase_flag_grp3_Training Type 2
0,C00001,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0,0,0
1,C00002,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0
2,C00003,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,1,0
3,C00004,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,C00005,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0


,ID_Customer,purchase_share_grp1_Automation Modules,purchase_share_grp1_Core Products,purchase_share_grp1_Maintenance Kits,purchase_share_grp1_Premium Products,purchase_share_grp1_Safety Solutions,purchase_share_grp1_Service Contracts,purchase_share_grp3_Calibration Kits Type 1,purchase_share_grp3_Calibration Kits Type 2,purchase_share_grp3_Compliance Safety Type 1,purchase_share_grp3_Compliance Safety Type 2,purchase_share_grp3_Controllers Type 1,purchase_share_grp3_Controllers Type 2,purchase_share_grp3_Core Advanced Type 1,purchase_share_grp3_Core Advanced Type 2,purchase_share_grp3_Core Basic Type 1,purchase_share_grp3_Core Basic Type 2,purchase_share_grp3_Core Replacement Type 1,purchase_share_grp3_Core Replacement Type 2,purchase_share_grp3_Extended Support Type 1,purchase_share_grp3_Extended Support Type 2,purchase_share_grp3_Facility Safety Type 2,purchase_share_grp3_Installation Type 1,purchase_share_grp3_Installation Type 2,purchase_share_grp3_Monitoring Type 1,purchase_share_grp3_Monitoring Type 2,purchase_share_grp3_Personal Safety Type 1,purchase_share_grp3_Personal Safety Type 2,purchase_share_grp3_Premium Custom Type 1,purchase_share_grp3_Premium Custom Type 2,purchase_share_grp3_Premium Plus Type 1,purchase_share_grp3_Premium Plus Type 2,purchase_share_grp3_Premium Standard Type 1,purchase_share_grp3_Premium Standard Type 2,purchase_share_grp3_Preventive Kits Type 1,purchase_share_grp3_Preventive Kits Type 2,purchase_share_grp3_Repair Kits Type 1,purchase_share_grp3_Repair Kits Type 2,purchase_share_grp3_Sensors Type 1,purchase_share_grp3_Sensors Type 2,purchase_share_grp3_Training Type 1,purchase_share_grp3_Training Type 2
0,C00001,0.19,0.17,0.17,0.48,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.17,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.27,0.00,0.00,0.21,0.00,0.00,0.17,0.00,0.19,0.00,0.00,0.00
1,C00002,0.33,0.33,0.00,0.19,0.14,0.00,0.00,0.00,0.00,0.00,0.00,0.19,0.00,0.14,0.10,0.10,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.14,0.00,0.00,0.00,0.00,0.00,0.10,0.10,0.00,0.00,0.00,0.00,0.14,0.00,0.00,0.00
2,C00003,0.00,0.33,0.12,0.15,0.15,0.25,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.09,0.24,0.00,0.00,0.00,0.00,0.00,0.15,0.09,0.03,0.00,0.00,0.00,0.00,0.00,0.00,0.05,0.00,0.00,0.10,0.05,0.00,0.00,0.07,0.00,0.00,0.13,0.00
3,C00004,0.26,0.48,0.26,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.26,0.00,0.00,0.24,0.00,0.24,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.26,0.00,0.00,0.00,0.00
4,C00005,0.15,0.22,0.00,0.15,0.00,0.47,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.22,0.00,0.00,0.00,0.00,0.00,0.00,0.33,0.14,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.15,0.00,0.00,0.00,0.00,0.00,0.00,0.15,0.00,0.00,0.00


Feature Quality Checks 

In [ ]:
feature_quality = pd.DataFrame({
    "table": [
        "dim_customer_features",
        "cust_prod_matrix_grp_1",
        "cust_prod_matrix_grp_3",
        "cust_prod_matrix_share_combined_grp_1_grp_3"],
    "rows": [
        len(dim_customer_features),
        len(cust_prod_matrix_grp_1),
        len(cust_prod_matrix_grp_3),
        len(cust_prod_matrix_share_combined_grp_1_grp_3)],
    "columns": [
        dim_customer_features.shape[1],
        cust_prod_matrix_grp_1.shape[1],
        cust_prod_matrix_grp_3.shape[1],
        cust_prod_matrix_share_combined_grp_1_grp_3.shape[1]],
    "missing_cells": [
        dim_customer_features.isna().sum().sum(),
        cust_prod_matrix_grp_1.isna().sum().sum(),
        cust_prod_matrix_grp_3.isna().sum().sum(),
        cust_prod_matrix_share_combined_grp_1_grp_3.isna().sum().sum()],
    "duplicate_customer_ids": [
        dim_customer_features["ID_Customer"].duplicated().sum(),
        cust_prod_matrix_grp_1["ID_Customer"].duplicated().sum(),
        cust_prod_matrix_grp_3["ID_Customer"].duplicated().sum(),
        cust_prod_matrix_share_combined_grp_1_grp_3["ID_Customer"].duplicated().sum()]})

display(feature_quality)

,table,rows,columns,missing_cells,duplicate_customer_ids
0,dim_customer_features,1993,67,0,0
1,cust_prod_matrix_grp_1,1993,7,0,0
2,cust_prod_matrix_grp_3,1993,36,0,0
3,cust_prod_matrix_share_combined_grp_1_grp_3,1993,42,0,0


Feature Dictionary 

In [ ]:
feature_dictionary = pd.DataFrame([
    {"feature": "Sales", "category": "Monetary", "description": "Total historical sales by customer."},
    {"feature": "Units", "category": "Volume", "description": "Total units purchased by customer."},
    {"feature": "distinct_products", "category": "Product breadth", "description": "Number of unique products purchased."},
    {"feature": "prd_grp_1_coverage_pct", "category": "Product coverage", "description": "Share of product group 1 categories purchased."},
    {"feature": "prd_grp_3_coverage_pct", "category": "Product coverage", "description": "Share of product group 3 categories purchased."},
    {"feature": "active_months", "category": "Activity", "description": "Number of months with at least one purchase."},
    {"feature": "lifetime_months", "category": "Tenure", "description": "Months between first and latest purchase."},
    {"feature": "months_since_last_purchase", "category": "Recency", "description": "Months since the latest purchase."},
    {"feature": "activity_ratio_lifetime", "category": "Activity", "description": "Active months divided by lifetime months."},
    {"feature": "avg_inactive_gap_months", "category": "Inactivity", "description": "Average months between purchase months."},
    {"feature": "historical_ltv", "category": "Value", "description": "Historical customer value proxy, equal to cumulative sales."},
    {"feature": "purchase_freq_per_year", "category": "Frequency", "description": "Annualized active purchase month frequency."},
    {"feature": "sf_total_events", "category": "Salesforce", "description": "Total Salesforce activity events."},
    {"feature": "sf_months_since_last_event", "category": "Salesforce", "description": "Months since latest Salesforce activity."},
    {"feature": "sales_without_recent_sf_activity", "category": "Sales activation", "description": "Customer bought recently but had no recent Salesforce activity."},
    {"feature": "sf_activity_without_recent_sales", "category": "Sales activation", "description": "Customer had recent Salesforce activity but no recent sales."}])

display(feature_dictionary)


,feature,category,description
0,Sales,Monetary,Total historical sales by customer.
1,Units,Volume,Total units purchased by customer.
2,distinct_products,Product breadth,Number of unique products purchased.
3,prd_grp_1_coverage_pct,Product coverage,Share of product group 1 categories purchased.
4,prd_grp_3_coverage_pct,Product coverage,Share of product group 3 categories purchased.
5,active_months,Activity,Number of months with at least one purchase.
6,lifetime_months,Tenure,Months between first and latest purchase.
7,months_since_last_purchase,Recency,Months since the latest purchase.
8,activity_ratio_lifetime,Activity,Active months divided by lifetime months.
9,avg_inactive_gap_months,Inactivity,Average months between purchase months.


Safe feature Outputs 

In [31]:
outputs = {
    "dim_customer_features": dim_customer_features,
    "cust_prod_matrix_grp_1": cust_prod_matrix_grp_1,
    "cust_prod_matrix_grp_3": cust_prod_matrix_grp_3,
    "cust_prod_matrix_grp_1_share": cust_prod_matrix_grp_1_share,
    "cust_prod_matrix_grp_3_share": cust_prod_matrix_grp_3_share,
    "cust_prod_matrix_share_combined_grp_1_grp_3": cust_prod_matrix_share_combined_grp_1_grp_3,
    "feature_quality": feature_quality,
    "feature_dictionary": feature_dictionary}

for name, df in outputs.items():
    df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

print("Saved files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)


Saved files:
- cust_prod_matrix_grp_1.csv
- cust_prod_matrix_grp_1_share.csv
- cust_prod_matrix_grp_3.csv
- cust_prod_matrix_grp_3_share.csv
- cust_prod_matrix_share_combined_grp_1_grp_3.csv
- dim_customer_features.csv
- feature_dictionary.csv
- feature_quality.csv
